# PasteTrace — Mamba Behavioral Sequence Model

**Yêu cầu:** Vào `Runtime → Change runtime type → T4 GPU` trước khi chạy.

Dataset: **205 sinh viên tổng hợp** trong `test_new_cohort/` (đã có sẵn trong repo — không cần upload Drive).

## Pipeline
```
Bước 0  Setup   (kiểm tra GPU, cài thư viện, clone repo)
Bước 1  Build   (meta.json → chuỗi sự kiện, data/sequences/)
Bước 2  Splits  (chia train 143 / val 31 / test 31, stratified)
Bước 3  Train   (Mamba model, lưu models/mamba/mamba.pt)
Bước 4  Test    (đánh giá held-out test set — chạy 1 lần duy nhất)
Bước 5  Lưu     (download mamba.pt về máy hoặc lên Drive)
```

## Bước 0a — Kiểm tra GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('GPU chua bat! Vao Runtime -> Change runtime type -> T4 GPU roi chay lai.')

## Bước 0b — Cài thư viện Mamba (~3 phút lần đầu)

In [ ]:
# causal-conv1d phai cai truoc mamba-ssm
# --no-build-isolation: cho build script thay duoc torch da co san
!pip install causal-conv1d --no-build-isolation -q
!pip install mamba-ssm --no-build-isolation -q
!pip install scikit-learn pandas plotly -q

# Xac nhan cai thanh cong
try:
    from mamba_ssm import Mamba
    print('OK — mamba-ssm installed successfully!')
except ImportError as e:
    print(f'FAILED: {e}')
    print('Thu chay lai cell nay mot lan nua, hoac restart runtime va chay lai tu dau.')

## Bước 0c — Clone repo từ GitHub

Lần đầu: clone repo về. Lần sau (runtime mới): clone lại hoặc pull update.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/lequocviet-3103/Fraud-Detection.git'
REPO_DIR = '/content/Fraud-Detection'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo da co, pull update...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())
!ls

## Bước 0d — Kiểm tra data

Dataset `test_new_cohort/` đã có sẵn trong repo (205 sinh viên tổng hợp — không cần upload Drive).

In [ ]:
import os

DATA_DIR = 'test_new_cohort'
if os.path.isdir(DATA_DIR):
    cases = sorted([d for d in os.listdir(DATA_DIR)
                    if os.path.isdir(os.path.join(DATA_DIR, d))])
    total = 0
    for c in cases:
        students = [s for s in os.listdir(os.path.join(DATA_DIR, c))
                    if os.path.isdir(os.path.join(DATA_DIR, c, s))]
        print(f'  {c}: {len(students)} students')
        total += len(students)
    print(f'\nOK — {total} students total in {len(cases)} cases')
else:
    print('CANH BAO: Khong tim thay test_new_cohort/. Chay lai Buoc 0c (git pull).')

## Bước 1 — Build Sequences

Đọc `meta.json` → vector sự kiện T/P/C → `data/sequences/`

Dùng flag `--data-dir test_new_cohort` để trỏ vào dataset tổng hợp.

In [ ]:
!python -m src.data.build_sequences --data-dir test_new_cohort --min-events 3

In [ ]:
import pandas as pd
df = pd.read_csv('data/sequences_index.csv')
print(df[['id','label','n_events','time_available']].to_string())
print(f'\nTong: {len(df)} sinh vien | cheat={sum(df.label==1)} | normal={sum(df.label==0)}')
print(f'Events: min={df.n_events.min()}  median={df.n_events.median():.0f}  max={df.n_events.max()}')

## Bước 2 — Make Splits

205 sinh viên → **train 143 / val 31 / test 31** (stratified 70/15/15, seed=42).

Dataset đủ lớn, dùng chế độ train/val/test thông thường (không cần LOO).

In [ ]:
!python -m src.data.make_splits --train 0.7 --val 0.15 --test 0.15 --seed 42

In [ ]:
import json
with open('data/splits.json') as f:
    sp = json.load(f)
for s in ['train', 'val', 'test']:
    c = sp['counts'][s]
    print(f"{s:6}: {c['total']:3d} samples  (cheat={c['cheat']}, normal={c['normal']})")

## Bước 3 — Train

Với 205 sinh viên, dùng **Cell A** (train/val/test thông thường).

Cell B (LOO) chỉ dùng khi dataset < 30 mẫu — không cần thiết ở đây.

In [ ]:
# Cell A: Train/Val/Test — dùng cho dataset 205 sinh vien nay
!python -m src.models.mamba_model train \
    --d-model 64 --n-layers 2 --dropout 0.2 \
    --epochs 80 --lr 1e-3 --patience 10 \
    --batch-size 8 --max-len 1000

In [ ]:
# Cell B: LOO (Leave-One-Out) — chi dung khi dataset < 30 mau
# Khong can thiet voi 205 sinh vien, de o day phong khi can
#
# !python -m src.models.mamba_model train --loo \
#     --d-model 64 --n-layers 2 \
#     --epochs 50 --lr 1e-3 --batch-size 4

In [ ]:
# Kiem tra model files
import os
for fname in ['mamba.pt', 'config.json', 'scaler.json']:
    p = f'models/mamba/{fname}'
    if os.path.isfile(p):
        print(f'OK     {p}  ({os.path.getsize(p)/1024:.1f} KB)')
    else:
        print(f'MISSING  {p}')

## Bước 4 — Test (chỉ chạy 1 lần cuối cùng)

⚠️ Đây là **held-out test set** — không dùng để điều chỉnh hyperparameter.

In [ ]:
!python -m src.models.mamba_model test

In [ ]:
# Hien thi ket qua
import json, pandas as pd
with open('results/mamba_metrics.json') as f:
    res = json.load(f)

m, mb = res['mamba'], res['majority_baseline']
rows = []
for name, mt in [('Mamba', m), ('Majority Baseline', mb)]:
    rows.append({'Model': name,
        'Accuracy':      f"{mt['accuracy']:.3f}",
        'Macro F1':      f"{mt['macro_f1']:.3f}",
        'Cheat F1':      f"{mt['cheat_f1']:.3f}",
        'Normal F1':     f"{mt['normal_f1']:.3f}",
        'Cheat Recall':  f"{mt['cheat_recall']:.3f}",
        'Normal Recall': f"{mt['normal_recall']:.3f}",
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Confusion matrix
import matplotlib.pyplot as plt, numpy as np
cm = np.array(m['confusion'])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred Normal', 'Pred Cheat'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['True Normal', 'True Cheat'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=18)
ax.set_title(f'Accuracy={m["accuracy"]:.3f}  Macro F1={m["macro_f1"]:.3f}')
plt.tight_layout()
plt.show()

## Bước 5 — Lưu model

In [ ]:
# Option A: Download ve may tinh (zip)
import zipfile
from google.colab import files

with zipfile.ZipFile('/content/mamba_trained.zip', 'w') as z:
    for f in ['models/mamba/mamba.pt', 'models/mamba/config.json',
              'models/mamba/scaler.json', 'results/mamba_metrics.json']:
        if os.path.isfile(f):
            z.write(f)
            print(f'Added {f}')

files.download('/content/mamba_trained.zip')
print('Downloading mamba_trained.zip ...')

In [ ]:
# Option B: Luu len Google Drive (de dung lai session sau)
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/PasteTrace/trained_models/mamba'
os.makedirs(SAVE_DIR, exist_ok=True)

for src in ['models/mamba/mamba.pt', 'models/mamba/config.json',
            'models/mamba/scaler.json', 'results/mamba_metrics.json']:
    if os.path.isfile(src):
        shutil.copy2(src, SAVE_DIR)
        print(f'Saved {os.path.basename(src)} -> Drive')

print(f'\nDone! Model o Drive: {SAVE_DIR}')

---
## Lần sau — Load model từ Drive (bỏ qua bước Train)

Khi mở Colab mới, chạy lại Bước 0a → 0b → 0c, rồi chạy cell này:

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
DRIVE_MODEL = '/content/drive/MyDrive/PasteTrace/trained_models/mamba'
os.makedirs('models/mamba', exist_ok=True)

for fname in ['mamba.pt', 'config.json', 'scaler.json']:
    shutil.copy2(f'{DRIVE_MODEL}/{fname}', f'models/mamba/{fname}')
    print(f'Loaded {fname} from Drive')

# Predict sinh vien moi
STUDENT_FOLDER = 'test_new_cohort/212/S01'  # thay bang folder thuc te
!python -m src.models.mamba_model predict {STUDENT_FOLDER}

---
## Lỗi thường gặp

| Lỗi | Fix |
|-----|-----|
| `RuntimeError: GPU chua bat` | Runtime → Change runtime type → T4 GPU |
| `Getting requirements to build wheel` | Dùng `--no-build-isolation` (Bước 0b đã sửa) |
| `mamba_ssm not found` | Chạy lại Bước 0b |
| `sequences_index.csv not found` | Chạy Bước 1 |
| `test_new_cohort/ not found` | Chạy lại Bước 0c (git pull) |
| `Cannot stratify split` | Dataset quá nhỏ — thêm data hoặc dùng LOO (Cell B Bước 3) |
| Session bị reset sau 12h | Chạy lại Bước 0a → 0c, load model từ Drive (cell cuối) |